Exploratory analysis of attestation aggregation propagation on Ethereum mainnet.

Recreates the aggregation analyses from the [ProbeLab/EthP2P HackMD](https://hackmd.io/rHVQ9iiRSwSiTdnVTkQM6g) and extends them with per-node, per-region, and per-client breakdowns.

In [ ]:
import os
from pathlib import Path

import clickhouse_connect
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")

SLOT_RANGE_START = 13601416
SLOT_RANGE_END = 13601485
NETWORK = "mainnet"

In [ ]:
sentries = clickhouse_connect.get_client(
    host=os.environ["CLICKHOUSE_HOST"],
    port=int(os.environ.get("CLICKHOUSE_PORT", 8443)),
    username=os.environ["CLICKHOUSE_USER"],
    password=os.environ["CLICKHOUSE_PASSWORD"],
    secure=True,
    autogenerate_session_id=False,
)


def query_sentries(sql: str) -> pd.DataFrame:
    result = sentries.query(sql)
    return pd.DataFrame(result.result_rows, columns=result.column_names)


print(f"Connected to {os.environ['CLICKHOUSE_HOST']}")

## Base query: per-aggregation propagation metrics

The core query from the HackMD doc. Groups by unique aggregation message and computes first-seen timing plus network-wide propagation percentiles (p50/p90/p95).

In [ ]:
BASE_QUERY = f"""
SELECT
    slot,
    committee_index,
    aggregator_index,
    bitCount(unhex(aggregation_bits)) AS agg_bits,
    message_id,
    min(slot_start_date_time)                                           AS slot_start_time,
    min(event_date_time)                                                AS agg_first_seen,
    dateDiff('millisecond', min(slot_start_date_time), min(event_date_time)) AS agg_first_seen_ms,
    dateDiff('millisecond', min(event_date_time), quantiles(0.50)(event_date_time)[1]) AS agg_p50_ms,
    dateDiff('millisecond', min(event_date_time), quantiles(0.90)(event_date_time)[1]) AS agg_p90_ms,
    dateDiff('millisecond', min(event_date_time), quantiles(0.95)(event_date_time)[1]) AS agg_p95_ms,
    count() AS observation_count
FROM default.libp2p_gossipsub_aggregate_and_proof
PREWHERE slot BETWEEN {SLOT_RANGE_START} AND {SLOT_RANGE_END}
WHERE meta_network_name = '{NETWORK}'
GROUP BY slot, committee_index, aggregator_index, agg_bits, message_id
ORDER BY slot, committee_index
"""

print(BASE_QUERY)

In [ ]:
df = query_sentries(BASE_QUERY)

df["agg_first_seen_s"] = df["agg_first_seen_ms"] / 1000.0
df["agg_p50_s"] = df["agg_p50_ms"] / 1000.0
df["agg_p50_arrival_s"] = df["agg_first_seen_s"] + df["agg_p50_s"]

print(f"Rows: {len(df):,}")
print(f"Slots: {df['slot'].nunique()}")
print(f"Unique aggregators: {df['aggregator_index'].nunique()}")
df.head()

## Aggregation first arrival

Distribution of when each unique aggregation was first observed by any sentry, measured in seconds from slot start. Color encodes the number of aggregated attestation bits.

In [ ]:
fig = px.histogram(
    df,
    x="agg_first_seen_s",
    color="agg_bits",
    nbins=120,
    range_x=[0, 14],
    labels={"agg_first_seen_s": "Seconds from slot start", "agg_bits": "Agg bits"},
)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    xaxis=dict(dtick=1),
    yaxis=dict(title="Aggregation count"),
    height=500,
    barmode="stack",
)
fig.show()

## First arrival shifted by p50 propagation

Approximates when the median network node sees each aggregation by adding the per-message p50 propagation delay to the first-arrival time. This better represents how the broader network experiences the message stream.

In [ ]:
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df["agg_first_seen_s"],
    nbinsx=120,
    name="First arrival",
    opacity=0.6,
    marker_color="#636EFA",
))

fig.add_trace(go.Histogram(
    x=df["agg_p50_arrival_s"],
    nbinsx=120,
    name="First arrival + p50",
    opacity=0.6,
    marker_color="#EF553B",
))

fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    barmode="overlay",
    xaxis=dict(title="Seconds from slot start", range=[0, 16], dtick=1),
    yaxis=dict(title="Aggregation count"),
    height=500,
)
fig.show()

## P50 propagation delay per aggregation

Each dot is a single aggregation message. Y-axis shows how long it took to reach 50% of the observing network after first being seen. Colored by first-arrival second to reveal whether early or late messages propagate differently.

In [ ]:
fig = px.scatter(
    df,
    x="agg_first_seen_s",
    y="agg_p50_s",
    color="agg_p50_s",
    color_continuous_scale="Plasma",
    opacity=0.4,
    labels={
        "agg_first_seen_s": "First arrival (s from slot start)",
        "agg_p50_s": "P50 propagation delay (s)",
    },
    hover_data={"slot": True, "aggregator_index": True, "observation_count": True},
)
fig.update_traces(marker=dict(size=3))
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    height=500,
    coloraxis_colorbar=dict(title="P50 (s)"),
)
fig.show()

## P50 propagation by slot

Distribution of p50 propagation delay across slots. Checks whether the delay is homogeneous or whether specific slots suffer worse propagation.

In [ ]:
fig = px.box(
    df,
    x="slot",
    y="agg_p50_ms",
    labels={"agg_p50_ms": "P50 propagation (ms)", "slot": "Slot"},
)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    xaxis=dict(tickangle=-90, tickfont=dict(size=8)),
    height=500,
)
fig.show()

## P50 propagation distribution

Histogram of all p50 propagation delays. The tail beyond 1 second highlights the aggregations that experience slow network spread.

In [ ]:
fig = px.histogram(
    df,
    x="agg_p50_s",
    nbins=100,
    labels={"agg_p50_s": "P50 propagation delay (s)"},
    color_discrete_sequence=["#AB63FA"],
)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    yaxis=dict(title="Aggregation count"),
    height=400,
)
fig.show()

print(f"P50 propagation delay stats (ms):")
print(df["agg_p50_ms"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_string())

## Aggregations vs next-slot block + data columns (first seen)

Overlays the aggregation first-arrival distribution (relative to their slot) with the next slot's block and data column first-seen times (shifted by +12s to align to the same slot's timeline). Tests whether bandwidth competition at the slot boundary delays aggregation propagation.

In [ ]:
BLOCK_QUERY = f"""
SELECT
    slot,
    dateDiff('millisecond', slot_start_date_time, event_date_time) AS first_seen_ms
FROM default.beacon_api_eth_v2_beacon_block
PREWHERE slot BETWEEN {SLOT_RANGE_START} AND {SLOT_RANGE_END + 1}
WHERE meta_network_name = '{NETWORK}'
"""

COLUMN_QUERY = f"""
SELECT
    slot,
    propagation_slot_start_diff AS first_seen_ms
FROM default.beacon_api_eth_v1_events_data_column_sidecar
PREWHERE slot BETWEEN {SLOT_RANGE_START} AND {SLOT_RANGE_END + 1}
WHERE meta_network_name = '{NETWORK}'
"""

df_blocks = query_sentries(BLOCK_QUERY)
df_columns = query_sentries(COLUMN_QUERY)

print(f"Block observations: {len(df_blocks):,}")
print(f"Column observations: {len(df_columns):,}")

In [ ]:
# Shift next-slot block/column times by +12s to align to previous slot's timeline
df_blocks_shifted = df_blocks.copy()
df_blocks_shifted["arrival_s"] = df_blocks_shifted["first_seen_ms"] / 1000.0 + 12.0
df_blocks_shifted["message_type"] = "Block (next slot)"

df_columns_shifted = df_columns.copy()
df_columns_shifted["arrival_s"] = df_columns_shifted["first_seen_ms"] / 1000.0 + 12.0
df_columns_shifted["message_type"] = "Data columns (next slot)"

df_agg_for_overlay = df[["agg_first_seen_s"]].copy()
df_agg_for_overlay.rename(columns={"agg_first_seen_s": "arrival_s"}, inplace=True)
df_agg_for_overlay["message_type"] = "Aggregation (first seen)"

df_overlay = pd.concat([
    df_agg_for_overlay[["arrival_s", "message_type"]],
    df_blocks_shifted[["arrival_s", "message_type"]],
    df_columns_shifted[["arrival_s", "message_type"]],
])

fig = px.histogram(
    df_overlay,
    x="arrival_s",
    color="message_type",
    nbins=200,
    barmode="overlay",
    opacity=0.5,
    range_x=[6, 18],
    labels={"arrival_s": "Seconds from slot start", "message_type": "Message type"},
    color_discrete_map={
        "Aggregation (first seen)": "#636EFA",
        "Block (next slot)": "#EF553B",
        "Data columns (next slot)": "#00CC96",
    },
)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    yaxis=dict(title="Message count"),
    height=500,
)
fig.show()

## Aggregations vs block + columns at p50

Same overlay but using the p50-adjusted arrival time for aggregations. Shows what the median network node experiences during the slot boundary transition.

In [ ]:
df_agg_p50_overlay = df[["agg_p50_arrival_s"]].copy()
df_agg_p50_overlay.rename(columns={"agg_p50_arrival_s": "arrival_s"}, inplace=True)
df_agg_p50_overlay["message_type"] = "Aggregation (p50 arrival)"

df_overlay_p50 = pd.concat([
    df_agg_p50_overlay[["arrival_s", "message_type"]],
    df_blocks_shifted[["arrival_s", "message_type"]],
    df_columns_shifted[["arrival_s", "message_type"]],
])

fig = px.histogram(
    df_overlay_p50,
    x="arrival_s",
    color="message_type",
    nbins=200,
    barmode="overlay",
    opacity=0.5,
    range_x=[6, 18],
    labels={"arrival_s": "Seconds from slot start", "message_type": "Message type"},
    color_discrete_map={
        "Aggregation (p50 arrival)": "#AB63FA",
        "Block (next slot)": "#EF553B",
        "Data columns (next slot)": "#00CC96",
    },
)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    yaxis=dict(title="Message count"),
    height=500,
)
fig.show()

---

# Extended analysis: per-node, per-region, per-client

Uses continuous `event_date_time` (no slot grouping) to show aggregation first-arrival patterns across the full observation period.

In [ ]:
import re

NODE_QUERY = f"""
SELECT
    event_date_time,
    propagation_slot_start_diff AS arrival_ms,
    slot,
    meta_client_name AS node,
    meta_client_implementation AS sentry_type,
    meta_client_geo_continent_code AS continent,
    meta_client_geo_country_code AS country
FROM default.libp2p_gossipsub_aggregate_and_proof
PREWHERE slot BETWEEN {SLOT_RANGE_START} AND {SLOT_RANGE_END}
WHERE meta_network_name = '{NETWORK}'
"""

CL_CLIENTS = ["prysm", "lighthouse", "teku", "nimbus", "lodestar", "grandine"]
CL_PATTERN = re.compile("|".join(CL_CLIENTS), re.IGNORECASE)

# Fallback: "tysm" sentries run prysm, "Xatu Sidecar (lighthouse)" runs lighthouse
SENTRY_TYPE_MAP = {"tysm": "prysm", "Xatu Sidecar (lighthouse)": "lighthouse"}


def extract_cl_client(node_name: str, sentry_type: str) -> str:
    match = CL_PATTERN.search(node_name)
    if match:
        return match.group(0).lower()
    return SENTRY_TYPE_MAP.get(sentry_type, "unknown")


print(NODE_QUERY)

In [ ]:
df_raw = query_sentries(NODE_QUERY)
df_raw["arrival_s"] = df_raw["arrival_ms"] / 1000.0
df_raw["event_date_time"] = pd.to_datetime(df_raw["event_date_time"])
df_raw["client"] = df_raw.apply(lambda r: extract_cl_client(r["node"], r["sentry_type"]), axis=1)

print(f"Raw observations: {len(df_raw):,}")
print(f"Unique nodes: {df_raw['node'].nunique()}")
print(f"Clients: {df_raw['client'].value_counts().to_dict()}")
print(f"Continents: {df_raw['continent'].value_counts().to_dict()}")
df_raw.head()

## Arrival time by node

Each point is a raw observation. X-axis is wall clock time, Y-axis is seconds from slot start. Faceted by node to compare how different sentries experience aggregation arrivals.

In [ ]:
# Top nodes by observation count for readable facets
top_nodes = df_raw["node"].value_counts().head(12).index.tolist()
df_top_nodes = df_raw[df_raw["node"].isin(top_nodes)].copy()

# Shorten node names for readability
df_top_nodes["node_short"] = df_top_nodes["node"].str.replace("ethpandaops/mainnet/", "", regex=False)

fig = px.scatter(
    df_top_nodes,
    x="event_date_time",
    y="arrival_s",
    color="node_short",
    opacity=0.15,
    labels={"event_date_time": "Time (UTC)", "arrival_s": "Seconds from slot start"},
)
fig.update_traces(marker=dict(size=2))
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    xaxis=dict(tickformat="%H:%M:%S"),
    height=600,
    legend=dict(font=dict(size=8)),
)
fig.show()

In [ ]:
# P50 arrival by node (aggregated)
node_stats = (
    df_raw
    .groupby("node")["arrival_ms"]
    .agg(["median", "mean", "count"])
    .sort_values("median", ascending=False)
    .head(20)
)
node_stats["node_short"] = node_stats.index.str.replace("ethpandaops/mainnet/", "", regex=False)

fig = px.bar(
    node_stats.reset_index(),
    x="node_short",
    y="median",
    color="count",
    color_continuous_scale="Plasma",
    labels={"median": "Median arrival (ms)", "node_short": "Node", "count": "Observations"},
)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=120),
    xaxis=dict(tickangle=-45, tickfont=dict(size=8)),
    height=500,
)
fig.show()

## Arrival time by continent

Aggregation arrival timing grouped by the observing node's geographic continent. Shows whether geographic distance from the aggregation source creates measurable propagation delays.

In [ ]:
fig = px.violin(
    df_raw[df_raw["continent"].isin(["EU", "NA", "AS", "OC"])],
    x="continent",
    y="arrival_s",
    color="continent",
    box=True,
    labels={"continent": "Continent", "arrival_s": "Seconds from slot start"},
)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    yaxis=dict(range=[6, 20]),
    showlegend=False,
    height=500,
)
fig.show()

In [ ]:
# Per-continent stats
continent_stats = (
    df_raw
    .groupby("continent")["arrival_ms"]
    .agg(["count", "median", "mean"])
    .sort_values("median", ascending=False)
)
continent_stats["median_s"] = continent_stats["median"] / 1000.0
continent_stats["mean_s"] = continent_stats["mean"] / 1000.0
print(continent_stats.to_string())

In [ ]:
# Scatter: continuous time by continent
df_cont = df_raw[df_raw["continent"].isin(["EU", "NA", "AS", "OC"])].copy()

fig = px.scatter(
    df_cont,
    x="event_date_time",
    y="arrival_s",
    color="continent",
    opacity=0.1,
    labels={"event_date_time": "Time (UTC)", "arrival_s": "Seconds from slot start"},
)
fig.update_traces(marker=dict(size=2))
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    xaxis=dict(tickformat="%H:%M:%S"),
    yaxis=dict(range=[6, 20]),
    height=500,
)
fig.show()

## Arrival time by client implementation

Compares arrival timing across different consensus client implementations used by the observing sentries. Differences here indicate client-specific gossipsub behavior or message processing pipelines.

In [ ]:
fig = px.violin(
    df_raw,
    x="client",
    y="arrival_s",
    color="client",
    box=True,
    labels={"client": "Client", "arrival_s": "Seconds from slot start"},
)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    yaxis=dict(range=[6, 20]),
    showlegend=False,
    height=500,
)
fig.show()

In [ ]:
# Per-client stats
client_stats = (
    df_raw
    .groupby("client")["arrival_ms"]
    .agg(["count", "median", "mean"])
    .sort_values("median", ascending=False)
)
client_stats["median_s"] = client_stats["median"] / 1000.0
client_stats["mean_s"] = client_stats["mean"] / 1000.0
print(client_stats.to_string())

In [ ]:
# Scatter: continuous time by client
fig = px.scatter(
    df_raw,
    x="event_date_time",
    y="arrival_s",
    color="client",
    opacity=0.1,
    labels={"event_date_time": "Time (UTC)", "arrival_s": "Seconds from slot start"},
)
fig.update_traces(marker=dict(size=2))
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    xaxis=dict(tickformat="%H:%M:%S"),
    yaxis=dict(range=[6, 20]),
    height=500,
)
fig.show()

## Heatmap: arrival time by node

Bucketed heatmap showing the density of aggregation arrivals per node over time. Each row is a node, each column is a time bucket.

In [ ]:
# Arrival time heatmap: bucket arrival_ms into 500ms bins, count per node
df_heat = df_raw.copy()
df_heat["arrival_bucket"] = (df_heat["arrival_ms"] // 500) * 500

top_nodes = df_heat["node"].value_counts().head(20).index.tolist()
df_heat = df_heat[df_heat["node"].isin(top_nodes)]
df_heat["node_short"] = df_heat["node"].str.replace("ethpandaops/mainnet/", "", regex=False)

heatmap_data = df_heat.groupby(["node_short", "arrival_bucket"]).size().reset_index(name="count")
heatmap_pivot = heatmap_data.pivot(index="node_short", columns="arrival_bucket", values="count").fillna(0)

# Normalize per-row (per node) to show relative distribution
heatmap_norm = heatmap_pivot.div(heatmap_pivot.sum(axis=1), axis=0)

fig = go.Figure(
    data=go.Heatmap(
        z=heatmap_norm.values,
        x=[f"{int(c)}ms" for c in heatmap_norm.columns],
        y=heatmap_norm.index.tolist(),
        colorscale="Plasma",
        hovertemplate="<b>Node:</b> %{y}<br><b>Bucket:</b> %{x}<br><b>Fraction:</b> %{z:.3f}<extra></extra>",
    )
)
fig.update_layout(
    margin=dict(l=200, r=30, t=30, b=60),
    xaxis=dict(title="Arrival time bucket"),
    yaxis=dict(autorange="reversed"),
    height=600,
)
fig.show()

## CDF: arrival time by client

Cumulative distribution function of arrival times, one line per client. Steeper curves = faster propagation to that client's nodes.

In [ ]:
fig = px.ecdf(
    df_raw,
    x="arrival_s",
    color="client",
    labels={"arrival_s": "Seconds from slot start", "client": "Client"},
)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    xaxis=dict(range=[6, 20]),
    yaxis=dict(title="CDF"),
    height=500,
)
fig.show()

## CDF: arrival time by continent

Same CDF but grouped by the observing node's continent.

In [ ]:
fig = px.ecdf(
    df_raw[df_raw["continent"].isin(["EU", "NA", "AS", "OC"])],
    x="arrival_s",
    color="continent",
    labels={"arrival_s": "Seconds from slot start", "continent": "Continent"},
)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    xaxis=dict(range=[6, 20]),
    yaxis=dict(title="CDF"),
    height=500,
)
fig.show()

---

# Late arrival pattern hunting

Focuses on aggregations with the highest delta between first-seen and p50 (the "spread"). Uses an enriched query that captures which node/region/client saw each message first and last.

In [ ]:
ENRICHED_QUERY = f"""
SELECT
    slot,
    message_id,
    aggregator_index,
    committee_index,
    bitCount(unhex(aggregation_bits)) AS agg_bits,
    count() AS observations,
    min(propagation_slot_start_diff) AS first_seen_ms,
    quantile(0.5)(propagation_slot_start_diff) AS p50_ms,
    quantile(0.5)(propagation_slot_start_diff) - min(propagation_slot_start_diff) AS spread_ms,
    quantile(0.9)(propagation_slot_start_diff) - min(propagation_slot_start_diff) AS spread_p90_ms,
    argMin(meta_client_name, propagation_slot_start_diff) AS first_observer_node,
    argMin(meta_client_geo_continent_code, propagation_slot_start_diff) AS first_observer_continent,
    argMin(meta_client_geo_country_code, propagation_slot_start_diff) AS first_observer_country,
    argMax(meta_client_name, propagation_slot_start_diff) AS last_observer_node,
    argMax(meta_client_geo_continent_code, propagation_slot_start_diff) AS last_observer_continent,
    argMax(meta_client_geo_country_code, propagation_slot_start_diff) AS last_observer_country
FROM default.libp2p_gossipsub_aggregate_and_proof
PREWHERE slot BETWEEN {SLOT_RANGE_START} AND {SLOT_RANGE_END}
WHERE meta_network_name = '{NETWORK}'
GROUP BY slot, message_id, aggregator_index, committee_index, agg_bits
ORDER BY spread_ms DESC
"""

df_msg = query_sentries(ENRICHED_QUERY)
df_msg["first_seen_s"] = df_msg["first_seen_ms"] / 1000.0
df_msg["spread_s"] = df_msg["spread_ms"] / 1000.0
df_msg["first_observer_client"] = df_msg.apply(
    lambda r: extract_cl_client(r["first_observer_node"], ""), axis=1
)
df_msg["last_observer_client"] = df_msg.apply(
    lambda r: extract_cl_client(r["last_observer_node"], ""), axis=1
)

p90_threshold = df_msg["spread_ms"].quantile(0.90)
df_msg["is_late"] = df_msg["spread_ms"] >= p90_threshold

print(f"Messages: {len(df_msg):,}")
print(f"Spread (ms) — median: {df_msg['spread_ms'].median():.0f}, "
      f"p90: {p90_threshold:.0f}, max: {df_msg['spread_ms'].max():.0f}")
print(f"Late messages (>= p90): {df_msg['is_late'].sum():,}")

## Spread vs first arrival

Each dot is one aggregation message. Highlights whether messages that arrive later into the slot also propagate more slowly, or whether the spread is independent of broadcast time.

In [ ]:
fig = px.scatter(
    df_msg,
    x="first_seen_s",
    y="spread_ms",
    color="is_late",
    color_discrete_map={True: "#EF553B", False: "#636EFA"},
    opacity=0.4,
    labels={
        "first_seen_s": "First seen (s from slot start)",
        "spread_ms": "Spread: p50 − first seen (ms)",
        "is_late": "Late (≥ p90)",
    },
    hover_data={"slot": True, "aggregator_index": True, "observations": True, "agg_bits": True},
)
fig.update_traces(marker=dict(size=3))
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    height=500,
)
fig.show()

## Spread by aggregator index

Are certain validators consistently producing aggregations that propagate slowly? Each dot is one message, grouped by its aggregator. Aggregators with many high-spread messages may have poor connectivity or be broadcasting late.

In [ ]:
# Per-aggregator stats
agg_stats = (
    df_msg
    .groupby("aggregator_index")
    .agg(
        msg_count=("spread_ms", "size"),
        median_spread=("spread_ms", "median"),
        mean_spread=("spread_ms", "mean"),
        p90_spread=("spread_ms", lambda x: x.quantile(0.9)),
        late_count=("is_late", "sum"),
        median_first_seen=("first_seen_ms", "median"),
    )
    .sort_values("median_spread", ascending=False)
)
agg_stats["late_pct"] = (agg_stats["late_count"] / agg_stats["msg_count"] * 100)

# Top 30 aggregators by median spread
top_agg = agg_stats.head(30).reset_index()

fig = px.bar(
    top_agg,
    x="aggregator_index",
    y="median_spread",
    color="late_pct",
    color_continuous_scale="Reds",
    hover_data={"msg_count": True, "late_count": True, "median_first_seen": ":.0f"},
    labels={
        "aggregator_index": "Aggregator index",
        "median_spread": "Median spread (ms)",
        "late_pct": "Late %",
    },
)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    xaxis=dict(type="category", tickfont=dict(size=7), tickangle=-90),
    height=500,
)
fig.show()

print(f"\nAggregators with >50% late messages:")
heavy_late = agg_stats[agg_stats["late_pct"] > 50]
print(f"  Count: {len(heavy_late)}")
if len(heavy_late) > 0:
    print(heavy_late[["msg_count", "median_spread", "late_pct", "median_first_seen"]].head(15).to_string())

In [ ]:
# Scatter: every aggregator's median spread vs median first-seen
# Reveals whether late-broadcast aggregators also have high spread
fig = px.scatter(
    agg_stats.reset_index(),
    x="median_first_seen",
    y="median_spread",
    size="msg_count",
    size_max=15,
    color="late_pct",
    color_continuous_scale="Reds",
    hover_data={"aggregator_index": True, "msg_count": True},
    labels={
        "median_first_seen": "Median first seen (ms from slot start)",
        "median_spread": "Median spread (ms)",
        "late_pct": "Late %",
    },
)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    height=500,
)
fig.show()

## Spread by first observer geography

Does the continent or country where a message was first seen predict how slowly it propagates to the rest of the network?

In [ ]:
fig = px.box(
    df_msg,
    x="first_observer_continent",
    y="spread_ms",
    color="is_late",
    color_discrete_map={True: "#EF553B", False: "#636EFA"},
    labels={
        "first_observer_continent": "First observer continent",
        "spread_ms": "Spread (ms)",
        "is_late": "Late (≥ p90)",
    },
    category_orders={"first_observer_continent": ["EU", "NA", "AS", "OC"]},
)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    height=500,
)
fig.show()

# Stats table
geo_stats = (
    df_msg
    .groupby("first_observer_continent")
    .agg(
        total=("spread_ms", "size"),
        median_spread=("spread_ms", "median"),
        mean_spread=("spread_ms", "mean"),
        late_count=("is_late", "sum"),
    )
    .sort_values("median_spread", ascending=False)
)
geo_stats["late_pct"] = geo_stats["late_count"] / geo_stats["total"] * 100
print(geo_stats.to_string())

## First observer → last observer flow

Sankey-style view: for late messages, which continent saw them first and which saw them last? Reveals directional propagation bottlenecks.

In [ ]:
# Cross-tabulation: first observer continent vs last observer continent (late only)
df_late = df_msg[df_msg["is_late"]].copy()

cross = pd.crosstab(
    df_late["first_observer_continent"],
    df_late["last_observer_continent"],
    margins=True,
)
print("Late messages: first observer (rows) → last observer (columns)")
print(cross.to_string())

# Heatmap
ct = pd.crosstab(
    df_late["first_observer_continent"],
    df_late["last_observer_continent"],
).reindex(index=["EU", "NA", "AS", "OC"], columns=["EU", "NA", "AS", "OC"], fill_value=0)

fig = go.Figure(
    data=go.Heatmap(
        z=ct.values,
        x=ct.columns.tolist(),
        y=ct.index.tolist(),
        colorscale="YlOrRd",
        text=ct.values,
        texttemplate="%{text}",
        hovertemplate="<b>First:</b> %{y}<br><b>Last:</b> %{x}<br><b>Count:</b> %{z}<extra></extra>",
    )
)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    xaxis=dict(title="Last observer continent"),
    yaxis=dict(title="First observer continent", autorange="reversed"),
    height=400,
    width=500,
)
fig.show()

## Spread by first observer client

Does the consensus client of the first observer correlate with propagation speed? If one client consistently appears as first-observer for high-spread messages, it might be reporting timestamps differently rather than truly seeing the message first.

In [ ]:
fig = px.box(
    df_msg,
    x="first_observer_client",
    y="spread_ms",
    color="first_observer_client",
    labels={
        "first_observer_client": "First observer client",
        "spread_ms": "Spread (ms)",
    },
)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    showlegend=False,
    height=500,
)
fig.show()

# Same for last observer
fig = px.box(
    df_msg,
    x="last_observer_client",
    y="spread_ms",
    color="last_observer_client",
    labels={
        "last_observer_client": "Last observer client",
        "spread_ms": "Spread (ms)",
    },
)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    showlegend=False,
    height=500,
)
fig.show()

# Client cross-tab for late messages
ct_client = pd.crosstab(
    df_late["first_observer_client"],
    df_late["last_observer_client"],
)
print("Late messages: first observer client (rows) → last observer client (columns)")
print(ct_client.to_string())

## Spread by aggregation bits and observation count

Does the number of attestation bits included in the aggregation (quality) or the number of sentry observations correlate with propagation spread?

In [ ]:
fig = px.scatter(
    df_msg,
    x="agg_bits",
    y="spread_ms",
    color="is_late",
    color_discrete_map={True: "#EF553B", False: "#636EFA"},
    opacity=0.4,
    labels={"agg_bits": "Aggregation bits", "spread_ms": "Spread (ms)", "is_late": "Late"},
    hover_data={"aggregator_index": True, "slot": True},
)
fig.update_traces(marker=dict(size=3))
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    height=450,
)
fig.show()

fig = px.scatter(
    df_msg,
    x="observations",
    y="spread_ms",
    color="is_late",
    color_discrete_map={True: "#EF553B", False: "#636EFA"},
    opacity=0.4,
    labels={"observations": "Sentry observations", "spread_ms": "Spread (ms)", "is_late": "Late"},
    hover_data={"aggregator_index": True, "slot": True},
)
fig.update_traces(marker=dict(size=3))
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    height=450,
)
fig.show()

LATENESS_QUERY = f"""
WITH msg_first AS (
    SELECT
        message_id,
        min(propagation_slot_start_diff) AS msg_first_seen_ms
    FROM default.libp2p_gossipsub_aggregate_and_proof
    PREWHERE slot BETWEEN {SLOT_RANGE_START} AND {SLOT_RANGE_END}
    WHERE meta_network_name = '{NETWORK}'
    GROUP BY message_id
)
SELECT
    a.meta_client_name AS node,
    a.meta_client_geo_continent_code AS continent,
    count() AS observations,
    avg(a.propagation_slot_start_diff - b.msg_first_seen_ms) AS avg_delay_from_first_ms,
    quantile(0.5)(a.propagation_slot_start_diff - b.msg_first_seen_ms) AS median_delay_ms,
    quantile(0.9)(a.propagation_slot_start_diff - b.msg_first_seen_ms) AS p90_delay_ms
FROM default.libp2p_gossipsub_aggregate_and_proof a
GLOBAL JOIN msg_first b ON a.message_id = b.message_id
PREWHERE a.slot BETWEEN {SLOT_RANGE_START} AND {SLOT_RANGE_END}
WHERE a.meta_network_name = '{NETWORK}'
GROUP BY node, continent
ORDER BY median_delay_ms DESC
"""

df_lateness = query_sentries(LATENESS_QUERY)
df_lateness["node_short"] = df_lateness["node"].str.replace("ethpandaops/mainnet/", "", regex=False)
df_lateness["client"] = df_lateness.apply(
    lambda r: extract_cl_client(r["node"], ""), axis=1
)

print(f"Nodes: {len(df_lateness)}")
df_lateness.head(10)

In [ ]:
LATENESS_QUERY = f"""
WITH msg_first AS (
    SELECT
        message_id,
        min(propagation_slot_start_diff) AS msg_first_seen_ms
    FROM default.libp2p_gossipsub_aggregate_and_proof
    PREWHERE slot BETWEEN {SLOT_RANGE_START} AND {SLOT_RANGE_END}
    WHERE meta_network_name = '{NETWORK}'
    GROUP BY message_id
)
SELECT
    a.meta_client_name AS node,
    a.meta_client_geo_continent_code AS continent,
    count() AS observations,
    avg(a.propagation_slot_start_diff - b.msg_first_seen_ms) AS avg_delay_from_first_ms,
    quantile(0.5)(a.propagation_slot_start_diff - b.msg_first_seen_ms) AS median_delay_ms,
    quantile(0.9)(a.propagation_slot_start_diff - b.msg_first_seen_ms) AS p90_delay_ms
FROM default.libp2p_gossipsub_aggregate_and_proof a
JOIN msg_first b ON a.message_id = b.message_id
PREWHERE a.slot BETWEEN {SLOT_RANGE_START} AND {SLOT_RANGE_END}
WHERE a.meta_network_name = '{NETWORK}'
GROUP BY node, continent
ORDER BY median_delay_ms DESC
"""

df_lateness = query_sentries(LATENESS_QUERY)
df_lateness["node_short"] = df_lateness["node"].str.replace("ethpandaops/mainnet/", "", regex=False)
df_lateness["client"] = df_lateness.apply(
    lambda r: extract_cl_client(r["node"], ""), axis=1
)

print(f"Nodes: {len(df_lateness)}")
df_lateness.head(10)

In [ ]:
fig = px.bar(
    df_lateness.sort_values("median_delay_ms", ascending=False),
    x="node_short",
    y="median_delay_ms",
    color="client",
    hover_data={"continent": True, "observations": True, "p90_delay_ms": ":.0f"},
    labels={
        "node_short": "Node",
        "median_delay_ms": "Median delay from first observer (ms)",
        "client": "Client",
    },
)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=120),
    xaxis=dict(tickangle=-45, tickfont=dict(size=7)),
    height=500,
)
fig.show()

# Same data as scatter: median delay vs p90 delay, colored by client
fig = px.scatter(
    df_lateness,
    x="median_delay_ms",
    y="p90_delay_ms",
    color="client",
    text="node_short",
    hover_data={"continent": True, "observations": True},
    labels={
        "median_delay_ms": "Median delay from first (ms)",
        "p90_delay_ms": "P90 delay from first (ms)",
        "client": "Client",
    },
)
fig.update_traces(textposition="top center", textfont_size=7)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    height=500,
)
fig.show()

## Committee index patterns

Are certain committees disproportionately represented among late-propagating aggregations?

In [ ]:
committee_stats = (
    df_msg
    .groupby("committee_index")
    .agg(
        total=("spread_ms", "size"),
        median_spread=("spread_ms", "median"),
        late_count=("is_late", "sum"),
    )
    .sort_values("median_spread", ascending=False)
)
committee_stats["late_pct"] = committee_stats["late_count"] / committee_stats["total"] * 100

fig = px.bar(
    committee_stats.reset_index(),
    x="committee_index",
    y="median_spread",
    color="late_pct",
    color_continuous_scale="Reds",
    labels={
        "committee_index": "Committee index",
        "median_spread": "Median spread (ms)",
        "late_pct": "Late %",
    },
)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    xaxis=dict(type="category", tickfont=dict(size=7)),
    height=450,
)
fig.show()

## Late arrivals summary matrix

Pivot of late-message rates across first-observer continent and first-observer client. Identifies which (region, client) combinations suffer the most propagation delay.

In [ ]:
# Late rate by (continent, client)
combo = (
    df_msg
    .groupby(["first_observer_continent", "first_observer_client"])
    .agg(total=("is_late", "size"), late=("is_late", "sum"), median_spread=("spread_ms", "median"))
)
combo["late_pct"] = combo["late"] / combo["total"] * 100

pivot_pct = combo["late_pct"].unstack(fill_value=0)
pivot_count = combo["total"].unstack(fill_value=0)

fig = go.Figure(
    data=go.Heatmap(
        z=pivot_pct.values,
        x=pivot_pct.columns.tolist(),
        y=pivot_pct.index.tolist(),
        colorscale="YlOrRd",
        text=[[f"{v:.0f}%\n(n={int(pivot_count.iloc[i, j])})"
               for j, v in enumerate(row)]
              for i, row in enumerate(pivot_pct.values)],
        texttemplate="%{text}",
        hovertemplate="<b>Continent:</b> %{y}<br><b>Client:</b> %{x}<br><b>Late %:</b> %{z:.1f}%<extra></extra>",
    )
)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    xaxis=dict(title="First observer client"),
    yaxis=dict(title="First observer continent", autorange="reversed"),
    height=350,
    width=600,
)
fig.show()

# Median spread version
pivot_spread = combo["median_spread"].unstack(fill_value=0)

fig = go.Figure(
    data=go.Heatmap(
        z=pivot_spread.values,
        x=pivot_spread.columns.tolist(),
        y=pivot_spread.index.tolist(),
        colorscale="Plasma",
        text=[[f"{int(v)}ms\n(n={int(pivot_count.iloc[i, j])})"
               for j, v in enumerate(row)]
              for i, row in enumerate(pivot_spread.values)],
        texttemplate="%{text}",
        hovertemplate="<b>Continent:</b> %{y}<br><b>Client:</b> %{x}<br><b>Median spread:</b> %{z:.0f} ms<extra></extra>",
    )
)
fig.update_layout(
    margin=dict(l=60, r=30, t=30, b=60),
    xaxis=dict(title="First observer client"),
    yaxis=dict(title="First observer continent", autorange="reversed"),
    height=350,
    width=600,
)
fig.show()